# Lab 20 - Platforms, Blueprints, agent identity, and registration

**Example: Google Vertex AI / Test V2.** Run the notebook in this order:

1. **Prerequisites:** existing login app and approved Graph scopes.
2. **Prep:** browser sign-in and all visible packages.
3. **Part 1:** platform inventory -> manually defined Blueprint groups -> one Blueprint and principal per approved group.
4. **Part 2:** selected agent's Package Details -> platform and source agent ID -> manually selected group -> Agent Identity.
5. **Part 3:** read/update a known registration, or explicitly create its first companion.

All Python and API calls are inline. There is no external workflow module.
Run one cell at a time in an ignored `workspace` copy with this lab's `.venv` kernel.
The inventory can contain many platforms; provisioning is limited to the groups
you explicitly configure and the one approved agent you select. Never create
objects for the entire inventory automatically.

This remains a non-production experiment. We retain the original Registry Sync
package and use a separate companion registration. We do not modify the provider
runtime, create connectors, configure agent credentials, or claim runtime
enforcement. Registration APIs are beta.

## Prerequisites

### App registration

Reuse the approved public-client login app in the intended tenant. Its
**Authentication > Mobile and desktop applications** platform must include
`http://localhost`. It is not a Web/SPA redirect. `AADSTS500113` means the app
has no registered reply address; ask the app owner to approve the missing setup.
Do not create an app, secret, or consent grant from this notebook.

The browser and Python kernel must run on the same local computer.
Sign in with the approved operator and complete the tenant's MFA requirements.
This uses MSAL browser sign-in, not device code.

### Scopes

These are delegated Graph permissions. They must already be approved; the
notebook does not grant them. Roles, ownership, licenses, and tenant policies
still apply.

| Phase | Read/reuse | Add only for explicitly enabled writes |
| --- | --- | --- |
| Prep / package discovery | `User.Read`, `CopilotPackages.Read.All` | None |
| Parts 1-2 / Entra objects | `User.Read`, `AgentIdentityBlueprint.Read.All`, `AgentIdentityBlueprintPrincipal.Read.All`, `AgentIdentity.Read.All` | `AgentIdentityBlueprint.Create`, `AgentIdentityBlueprintPrincipal.Create`, `AgentIdentity.Create.All` |
| Part 3 / registration | Earlier reads plus `AgentRegistration.Read.All` | `AgentRegistration.ReadWrite.All` |
| Optional cleanup | Earlier reads | `AgentIdentity.DeleteRestore.All` or `AgentIdentityBlueprint.DeleteRestore.All`; registration deletion uses `AgentRegistration.ReadWrite.All` |

Stop at any new consent or production boundary. Tokens stay in kernel memory;
real IDs and responses stay in ignored local files. Never print tokens or paste
IDs into tracked cells. Existing Registry Sync imports are prerequisites; no new
connection or sync is triggered here.

## Prep

### Local configuration and imports

Configure every target, approval, create switch, and cleanup switch together at
the beginning of `setup-code`. Keep `RUN_WRITES = False` while inspecting; all
mutation controls are off by default. A small HTTP helper saves returned IDs and blocks
an unresolved/repeated write; it is not a production workflow engine.

Routine writes do not ask for `APPLY` unless you set `CONFIRM_EACH_WRITE = True`.
The write switches, group approvals, and failed-write guard still apply.
Deletion always requires `APPLY`; first-time registration still requires `FIRST`.

The same `evidence/trial-2-jupyter-notebook/state.json` is reused so existing IDs
are not lost. Part 1 can import a prior single-group Blueprint binding from that
state. Do not delete state to force another create. The older helper-based state
and HTTP evidence remain separate and untouched.

In [ ]:
import json
from datetime import datetime
from time import time
from getpass import getpass
from pathlib import Path
from urllib.parse import quote, urlsplit
import requests
import msal

# Configure the complete run here before executing any later cell.
TARGET_NAME = 'Test V2'
PLATFORM = 'GoogleVertexAI'
REGISTRY_SYNC_PLATFORMS = {'GoogleVertexAI'}
BLUEPRINT_GROUPS = {'GoogleVertexAI': ['test-v2-dev']}
SELECTED_BLUEPRINT_GROUP = 'test-v2-dev'

RUN_WRITES = False
CONFIRM_EACH_WRITE = False
GROUPS_APPROVED = False
AGENT_GROUP_APPROVED = False
CREATE_BLUEPRINT = False
CREATE_PRINCIPAL = False
CREATE_IDENTITY = False
CREATE_COMPANION = False

DELETE_OBJECT = None
CONFIRMED_NO_DEPENDENTS = False
CLEANUP_PLATFORM = None
CLEANUP_GROUP = None

GRAPH = 'https://graph.microsoft.com'
PACKAGES = '/v1.0/copilot/admin/catalog/packages'
REGISTRATIONS = '/beta/copilot/agentRegistrations'

folders = [Path.cwd(), *Path.cwd().parents]
candidates = [p for folder in folders for p in (folder, folder / 'lab-20-registry-sync-identity-gaps')]
LAB_ROOT = next(p for p in candidates if (p / 'identity-assignment-research.md').is_file())
PRIVATE = LAB_ROOT / 'evidence' / 'trial-2-jupyter-notebook'
PRIVATE.mkdir(parents=True, exist_ok=True)
STATE_FILE = PRIVATE / 'state.json'
state = json.loads(STATE_FILE.read_text(encoding='utf-8')) if STATE_FILE.exists() else {}
access_token = None
auth_client = None
auth_account = None
access_token_scopes = set()
access_token_expires_at = 0

def save_state():
    temporary = STATE_FILE.with_suffix('.tmp')
    temporary.write_text(json.dumps(state, indent=2), encoding='utf-8')
    temporary.replace(STATE_FILE)

def request_graph(method, path, body=None, *, record=None, expected=None):
    url = GRAPH + path if path.startswith('/') else path
    parsed = urlsplit(url)
    assert parsed.scheme == 'https' and parsed.hostname == 'graph.microsoft.com', 'Unexpected Graph host.'
    assert not parsed.username and not parsed.password and parsed.port in (None, 443), 'Unexpected Graph authority.'
    assert access_token, 'Sign in first.'
    writing = method != 'GET'
    if writing:
        if not RUN_WRITES:
            print({'method': method, 'status': 'skipped - RUN_WRITES is False'})
            return None
        assert GROUPS_APPROVED, 'Approve the configured Blueprint groups in Part 1 first.'
        assert record and record not in state.get('completed_writes', []), 'Write already recorded; read its result instead of repeating it.'
        assert not state.get('pending_write'), 'An earlier write is unresolved. Inspect private state; do not resend it.'
        if method == 'DELETE' or CONFIRM_EACH_WRITE:
            assert getpass(f'Type APPLY to approve this {method} ({record}): ') == 'APPLY', 'Not approved.'
        state['pending_write'] = {'method': method, 'path': path, 'record': record}
        save_state()
    try:
        response = requests.request(method, url, headers={'Authorization': 'Bearer ' + access_token, 'Accept': 'application/json', 'OData-Version': '4.0'}, json=body, timeout=60, allow_redirects=False)
    except requests.RequestException:
        raise RuntimeError('Network error. If this was a write, its outcome is unknown; do not repeat it.') from None
    allowed = expected or {'GET': (200,), 'POST': (201,), 'PATCH': (200,), 'DELETE': (204,)}[method]
    print({'method': method, 'http_status': response.status_code})
    assert response.status_code in allowed, f'HTTP {response.status_code}; stop and resolve the error. No automatic retry.'
    if response.status_code == 404:
        return None
    try:
        data = response.json() if response.content else {}
    except requests.exceptions.JSONDecodeError:
        raise RuntimeError('Response was not JSON. A write may have succeeded; inspect before retrying.') from None
    assert isinstance(data, dict), 'Unexpected response shape.'
    if writing:
        if method == 'POST':
            assert data.get('id'), 'Create returned no ID; its outcome needs investigation.'
            state.setdefault('created_here', {})[record] = data['id']
        state[record] = data
        state.setdefault('completed_writes', []).append(record)
        state.pop('pending_write')
        save_state()
    return data

print({'target': TARGET_NAME, 'platform': PLATFORM, 'writes_enabled': RUN_WRITES})


### Sign in

Enter the existing tenant/client IDs privately, choose the approved account once,
and finish any MFA. Prep requests the permissions needed by the main notebook
flow according to `RUN_WRITES`; later phases reuse that token without reopening
the browser. A new scope, expiry, or Entra policy can still require interaction.
No device code or Tk popup is used, and a passkey enrollment prompt is controlled
by Entra policy rather than this notebook.

In [ ]:
state['tenant_id'] = state.get('tenant_id') or getpass('Approved tenant ID: ').strip()
state['client_id'] = state.get('client_id') or getpass('Existing approved public-client ID: ').strip()
assert state['tenant_id'] and state['client_id'], 'Tenant and client are required.'
save_state()
DISCOVERY_SCOPES = ['User.Read', 'CopilotPackages.Read.All']
ENTRA_READ_SCOPES = ['User.Read', 'AgentIdentityBlueprint.Read.All', 'AgentIdentityBlueprintPrincipal.Read.All', 'AgentIdentity.Read.All']
ENTRA_CREATE_SCOPES = ['AgentIdentityBlueprint.Create', 'AgentIdentityBlueprintPrincipal.Create', 'AgentIdentity.Create.All']
REGISTRATION_READ_SCOPES = ['AgentRegistration.Read.All']
SESSION_SCOPES = DISCOVERY_SCOPES + ENTRA_READ_SCOPES + REGISTRATION_READ_SCOPES
if RUN_WRITES:
    SESSION_SCOPES += ENTRA_CREATE_SCOPES + ['AgentRegistration.ReadWrite.All']

def sign_in(scopes):
    global access_token, auth_client, auth_account, access_token_scopes, access_token_expires_at
    requested = {scope.lower() for scope in scopes}
    if access_token and requested <= access_token_scopes and time() < access_token_expires_at - 60:
        print({'signed_in': 'reused', 'permission_count': len(requested)})
        return
    if auth_client is None:
        auth_client = msal.PublicClientApplication(state['client_id'], authority='https://login.microsoftonline.com/' + state['tenant_id'], exclude_scopes=['offline_access'])
    full_scopes = [GRAPH + '/' + scope for scope in sorted(set(scopes))]
    result = auth_client.acquire_token_silent(full_scopes, account=auth_account) if auth_account else None
    mode = 'silent'
    if not result:
        mode = 'interactive'
        access_token = None
        access_token_scopes = set()
        access_token_expires_at = 0
        options = {'scopes': full_scopes, 'timeout': 300}
        if auth_account and auth_account.get('username'):
            options['login_hint'] = auth_account['username']
        else:
            options['prompt'] = msal.Prompt.SELECT_ACCOUNT
        print('Opening Microsoft Entra sign-in in your browser. Complete sign-in there, including any required MFA or consent.')
        result = auth_client.acquire_token_interactive(**options)
    if not result.get('access_token'):
        access_token = None
        access_token_scopes = set()
        access_token_expires_at = 0
        auth_account = None
        auth_client = None
        raise RuntimeError('Browser sign-in did not complete. Check the browser message and the approved http://localhost desktop redirect URI; token responses are not displayed.')
    access_token = result['access_token']
    granted = {scope.rsplit('/', 1)[-1].lower() for scope in result['scope'].split()} if result.get('scope') else requested.copy()
    if not requested <= granted:
        access_token = None
        access_token_scopes = set()
        access_token_expires_at = 0
        auth_account = None
        auth_client = None
        raise RuntimeError('The token does not contain every requested permission.')
    access_token_scopes = granted
    access_token_expires_at = int(result.get('expires_on') or time() + int(result.get('expires_in', 300)))
    operator = request_graph('GET', '/v1.0/me')
    if state.get('operator_id') and state['operator_id'] != operator['id']:
        access_token = None
        access_token_scopes = set()
        access_token_expires_at = 0
        auth_account = None
        auth_client = None
        raise RuntimeError('Operator changed; stop and review ownership.')
    claims = result.get('id_token_claims', {})
    username = claims.get('preferred_username')
    accounts = auth_client.get_accounts(username=username) if username else auth_client.get_accounts()
    selected_oid = claims.get('oid')
    auth_account = next((account for account in accounts if account.get('local_account_id') == selected_oid), accounts[0] if len(accounts) == 1 else auth_account)
    state['operator_id'] = operator['id']
    save_state()
    print({'signed_in': mode, 'permission_count': len(requested)})

sign_in(SESSION_SCOPES)


### Get all packages

Follow every page returned by `GET /v1.0/copilot/admin/catalog/packages`.
"All" means visible to this signed-in operator, not a guaranteed complete tenant
inventory. Names identify candidates, not unique agents.

In [ ]:
def list_packages():
    packages, seen = [], set()
    url = GRAPH + PACKAGES
    while url:
        assert url not in seen and len(seen) < 1000, 'Repeated or excessive pagination.'
        assert urlsplit(url).path == PACKAGES, 'Unexpected pagination resource.'
        seen.add(url)
        page = request_graph('GET', url)
        packages.extend(page['value'])
        url = page.get('@odata.nextLink')
    return packages


packages = list_packages()
print({'visible_packages': len(packages)})


## Part 1 - Create or reuse Blueprints by platform

### 1.1 Group packages by platform

Use the List packages results already loaded in Prep. Each item supplies `id`
and `platform`, so this cell groups them and prints counts without any new
API requests. Missing platform values remain explicitly labeled.

This is a platform inventory, not proof of Registry Sync origin. A third-party
label alone does not prove Registry Sync origin: a manually created companion
can have the same platform.

Use approved setup/portal records to confirm the relevant third-party Registry
Sync platforms in 1.2. Package Details and source-metadata inspection are deferred
to Part 2 for the selected agent; there is no per-package detail scan here.

In [ ]:
packages_by_platform = {}
for package in packages:
    platform = package.get('platform') or '(platform missing)'
    packages_by_platform.setdefault(platform, []).append(package)
for platform, members in sorted(packages_by_platform.items()):
    print({'platform': platform, 'packages': len(members)})


### 1.2 Identify Blueprint groups per platform

Review the platform counts above and approved setup/portal records.
`REGISTRY_SYNC_PLATFORMS` is your manually confirmed set of
third-party Registry Sync platform labels, not an automatic conclusion from a
name. The default is the already approved Google Vertex AI example. Other
providers need independent confirmation; their presence in the list is not proof.

Edit `BLUEPRINT_GROUPS` to list the groups to provision **per platform**. Start
with `test-v2-dev` only. An additional group must have an approved non-production
purpose, responsible owner, credential controller, and compatible shared access.
`GROUPS_APPROVED = True` approves that explicit group plan, not every agent on
those platforms. Group labels contain no colons.

This notebook intentionally scopes group labels by platform. Connections are
provenance only: one connection can supply agents to different groups, and one
group can contain approved agents from different connections.

The saved `blueprint_bindings[platform][group]` points to the actual Blueprint and
principal records. This is the lookup Part 2 will use. It is not an Entra group.

For a new run, the sponsor defaults to the signed-in operator without a prompt.
A valid saved sponsor override is preserved. Approve this choice with the group
plan. To use another approved sponsor, set `state['sponsor_id']` privately before
running this cell; do not put real IDs into tracked cells.

An override must be an approved user's **Object ID** from Entra admin center >
Users > the sponsor > Overview. This notebook uses `/users/{id}` bindings, not
app/client IDs, tenant IDs, or placeholders such as `tbc`. Invalid saved IDs
prompt again. GUID validation checks format only, not whether the user exists
or is approved as sponsor. A failed-write marker is never cleared by this cell.

In [ ]:
from uuid import UUID

# 1. Validate the configured platform/group plan against the listed inventory.
assert REGISTRY_SYNC_PLATFORMS <= packages_by_platform.keys(), 'A selected platform is not in this visible inventory.'
assert '(platform missing)' not in REGISTRY_SYNC_PLATFORMS, 'A missing platform cannot be selected for provisioning.'
assert BLUEPRINT_GROUPS.keys() <= REGISTRY_SYNC_PLATFORMS, 'Review each configured platform as third-party Registry Sync first.'
for platform, groups in BLUEPRINT_GROUPS.items():
    assert groups and len(groups) == len(set(groups)), 'Each platform needs distinct group labels.'
    assert all(group.strip() and ':' not in group for group in groups) and ':' not in platform, 'Use nonempty labels without colons.'
    print({'platform': platform, 'groups': groups, 'approved': GROUPS_APPROVED})

# 2. Load group bindings and preserve any Blueprint saved by an earlier single-group run.
# Conflicting mappings must be resolved instead of creating replacement objects.
bindings = state.setdefault('blueprint_bindings', {})
if state.get('blueprint'):
    legacy_platform = state.get('original_before', {}).get('platform')
    legacy_group = state.get('blueprint_group')
    assert legacy_platform and legacy_group, 'Existing Blueprint needs its original platform/group mapping; do not create a replacement.'
    existing = bindings.setdefault(legacy_platform, {}).get(legacy_group)
    if existing:
        assert state[existing['blueprint_record']]['id'] == state['blueprint']['id'], 'Existing group bindings conflict.'
    else:
        bindings[legacy_platform][legacy_group] = {'blueprint_record': 'blueprint', 'principal_record': 'principal'}

# 3. Preserve a saved sponsor; otherwise default to the signed-in operator (no API write).
sponsor_id = state.get('sponsor_id') or state['operator_id']
try:
    sponsor_object_id = UUID(sponsor_id) if isinstance(sponsor_id, str) else None
except ValueError:
    sponsor_object_id = None
if sponsor_object_id is None:
    if sponsor_id:
        print('Previously saved sponsor ID was invalid. Enter a replacement in the private prompt below.')
    try:
        sponsor_object_id = UUID(getpass('Approved sponsor user Object ID: ').strip())
    except ValueError:
        raise ValueError('Sponsor must be an Entra user Object ID (GUID), not a placeholder.') from None
state['sponsor_id'] = str(sponsor_object_id)
save_state()
print({'sponsor_id_format_valid': True, 'sponsor_saved_locally': True})


### 1.3 Create or reuse one Blueprint and principal per group

The loop covers only `BLUEPRINT_GROUPS`. Saved IDs are reused automatically.
With `CREATE_BLUEPRINT = True`, no existing-ID prompt is shown. Leave it False
to be prompted for an existing Blueprint's **object ID** when none is saved.
Enable `CREATE_BLUEPRINT` / `CREATE_PRINCIPAL`
only after confirming that the corresponding objects do not already exist.
A missing/denied read or lost ID is not proof that an object is absent.

For each group, read/create the Blueprint application, then read/create its
Blueprint principal using the documented `appId` lookup. Both are needed before
creating a child Agent Identity. The saved Blueprint has **object `id`** (for
application GET/DELETE) and **`appId`** (the parent identifier used in Part 2).
The principal has its own service-principal ID. No credentials are created.

In [ ]:
from uuid import UUID

# 1. Use the top-level creation switches and sign in with the approved Entra permissions.
entra_scopes = ENTRA_READ_SCOPES.copy()
if RUN_WRITES:
    entra_scopes += ENTRA_CREATE_SCOPES
sign_in(entra_scopes)

# 2. Process only configured groups, keeping stable keys for each group's saved objects.
ready_groups = set()
for platform, groups in BLUEPRINT_GROUPS.items():
    for group in groups:
        binding = bindings.setdefault(platform, {}).setdefault(group, {
            'blueprint_record': f'blueprint:{platform}:{group}',
            'principal_record': f'principal:{platform}:{group}',
        })
        bp_record, principal_record = binding['blueprint_record'], binding['principal_record']
        save_state()
        # 3. Reuse a Blueprint by its application object ID, or explicitly create one.
        bp = state.get(bp_record)
        object_id = bp['id'] if bp else ''
        if not object_id and not CREATE_BLUEPRINT:
            object_id = getpass(f'Existing Blueprint object ID for {platform}/{group}, or blank to leave unresolved: ').strip()
        if object_id:
            bp = request_graph('GET', '/v1.0/applications/' + quote(object_id, safe='') + '/microsoft.graph.agentIdentityBlueprint')
            assert bp['id'] == object_id, 'Unexpected Blueprint object.'
        elif CREATE_BLUEPRINT:
            bp = request_graph('POST', '/v1.0/applications/microsoft.graph.agentIdentityBlueprint', {
                'displayName': f'{platform} - {group} - disposable Blueprint',
                'sponsors@odata.bind': [GRAPH + '/v1.0/users/' + str(UUID(state['sponsor_id']))],
                'owners@odata.bind': [GRAPH + '/v1.0/users/' + quote(state['operator_id'], safe='')],
            }, record=bp_record)
        principal = None
        if bp:
            # 4. Prevent different groups from sharing the same Blueprint, then save it.
            assert bp.get('appId'), 'Blueprint appId is required.'
            for platform_bindings in bindings.values():
                for other in platform_bindings.values():
                    other_bp = state.get(other['blueprint_record'])
                    assert other['blueprint_record'] == bp_record or not other_bp or other_bp['appId'] != bp['appId'], 'The same Blueprint is already bound to another group.'
            state[bp_record] = bp
            save_state()
            # 5. Find/create the principal using Blueprint appId, not its application object ID.
            app_id = str(UUID(bp['appId']))
            principal = request_graph('GET', f"/v1.0/servicePrincipals(appId='{app_id}')/microsoft.graph.agentIdentityBlueprintPrincipal", expected=(200, 404))
            if principal is None and CREATE_PRINCIPAL:
                assert not state.get(principal_record), 'A known principal is unavailable; do not recreate it.'
                principal = request_graph('POST', '/v1.0/servicePrincipals/microsoft.graph.agentIdentityBlueprintPrincipal', {'appId': app_id}, record=principal_record)
            if principal:
                # 6. Mark the group ready for Part 2 only after confirming a matching, enabled principal.
                assert principal.get('appId') == bp['appId'] and principal.get('accountEnabled') is not False, 'Principal mismatch or disabled principal.'
                state[principal_record] = principal
                save_state()
                ready_groups.add((platform, group))
        print({'platform': platform, 'group': group, 'blueprint_available': bool(bp), 'principal_available': bool(principal)})


## Part 2 - Create or reuse an Agent Identity for the selected agent

### 2.1 Get the agent's Package Details, platform, and source agent ID

Read details only for the matching `TARGET_NAME` / `PLATFORM` candidates, then
select the approved original. Other inventory packages are not inspected.
Source-metadata parsing starts here, not in Part 1. This GCP example requires
the observed Registry Sync markers and source scope; other provider shapes need
a separately reviewed adaptation, not a guessed interpretation.

The authoritative platform for the following lookup is **`detail['platform']`**.
The exact source agent ID comes from the parsed definition. Connection ID is
recorded only as sync provenance. Do not use it to discover platform or group.

If several originals share the name, choose the approved Package ID privately
from `candidates.json`. Saved source mappings cannot silently switch to another
agent. For a different agent, use a separately reviewed private state file and
recover existing IDs before provisioning; this is not a fleet loop.

For local inspection only, run `print(json.dumps(detail, indent=2))` in your
**ignored working copy** and clear its output before sharing.

In [ ]:
# 1. Decode embedded definition JSON and track malformed entries instead of guessing source IDs.
def source_definitions(detail):
    definitions, invalid = [], 0
    sections = detail.get('elementDetails') or []
    if not isinstance(sections, list):
        return [], 1
    for section in sections:
        if not isinstance(section, dict):
            invalid += 1
            continue
        elements = section.get('elements') or []
        if not isinstance(elements, list):
            invalid += 1
            continue
        for element in elements:
            if not isinstance(element, dict):
                invalid += 1
                continue
            raw = element.get('definition')
            if not raw:
                continue
            if not isinstance(raw, str):
                invalid += 1
                continue
            try:
                parsed = json.loads(raw)
            except json.JSONDecodeError:
                invalid += 1
                continue
            if isinstance(parsed, dict) and parsed.get('SourceAgentId'):
                definitions.append(parsed)
    return definitions, invalid

# 2. Require exactly one usable source definition before trusting the agent metadata.
def read_source(detail):
    definitions, invalid = source_definitions(detail)
    assert not invalid and len(definitions) == 1, 'Source metadata is missing, malformed, or ambiguous; inspect privately.'
    return definitions[0]

# 3. Inspect only the requested agent's candidates and identify originals using the observed GCP markers.
candidates = [p for p in packages if p.get('platform') == PLATFORM and p.get('displayName') == TARGET_NAME]
originals = []
for candidate in candidates:
    detail = request_graph('GET', PACKAGES + '/' + quote(candidate['id'], safe=''))
    source = read_source(detail)
    source_ids = source.get('SourceIds', {})
    if source_ids.get('ConnectionId') and source_ids.get('mac.agentRegistrationType') == 'ConnectedPlatform':
        assert source_ids.get('mac.agentRegistrationProviderType') == PLATFORM, 'Provider markers disagree.'
        assert detail['id'] == candidate['id'] and detail.get('displayName') == TARGET_NAME and detail.get('platform') == PLATFORM, 'Candidate changed.'
        originals.append(detail)
# 4. Keep candidate details private and require a manual choice if several originals match.
(PRIVATE / 'candidates.json').write_text(json.dumps(originals, indent=2), encoding='utf-8')
assert originals, 'No Registry Sync Test V2 candidate found. Do not create anything.'
if len(originals) > 1:
    chosen_id = getpass('Approved original Package ID from private candidates.json: ')
    originals = [d for d in originals if d['id'] == chosen_id]
assert len(originals) == 1, 'Select exactly one approved original.'
# 5. Read platform directly from the package and preserve the exact provider source ID and scope.
original = detail = originals[0]
agent_platform = detail['platform']
source = read_source(original)
source_ids = source['SourceIds']
assert source_ids.get('mac.projectId') and source_ids.get('mac.region'), 'Source scope is incomplete.'
# 6. Prevent an accidental agent switch and retain the first baseline for later comparisons.
assert not state.get('original_id') or state['original_id'] == original['id'], 'Saved original differs; reconcile first.'
assert not state.get('source_agent_id') or state['source_agent_id'] == source['SourceAgentId'], 'Saved source differs.'
state.update(agent_platform=agent_platform, original_id=original['id'], source_agent_id=source['SourceAgentId'], connection_id=source_ids['ConnectionId'])
state.setdefault('original_before', original)
save_state()
print({'original_selected': True, 'connection_present': True, 'project_present': True, 'region_present': True, 'original_identity_present': bool(original.get('agentIdentityId'))})


### 2.2 Manually choose the agent's Blueprint group and obtain its Blueprint

Use the package's platform to find the groups created/reused in Part 1.
Choose one group with `SELECTED_BLUEPRINT_GROUP` in the Prep configuration.
Set `AGENT_GROUP_APPROVED` there only after approving this particular source
agent's membership.

`bindings[agent_platform][blueprint_group]` resolves the saved Blueprint.
`blueprint['appId']` is the Blueprint ID supplied to the Agent Identity API.
An existing group's label cannot silently reparent an existing agent.

In [ ]:
# 1. Use the group and separate membership approval configured at the top.
blueprint_group = SELECTED_BLUEPRINT_GROUP
# 2. Require a prepared group from Part 1 and reject an implicit change to an existing mapping.
assert agent_platform in BLUEPRINT_GROUPS, 'Configure this returned platform in Part 1 first.'
assert blueprint_group in BLUEPRINT_GROUPS[agent_platform], 'Choose a configured group for this platform.'
assert (agent_platform, blueprint_group) in ready_groups, 'Blueprint and principal are not ready; resolve Part 1 before continuing.'
assert not state.get('blueprint_group') or state['blueprint_group'] == blueprint_group, 'An existing agent/group mapping requires a separate migration decision.'
# 3. Resolve the saved Blueprint/principal using package platform plus the chosen group, not connection ID.
binding = bindings[agent_platform][blueprint_group]
blueprint_record = binding['blueprint_record']
blueprint = state.get(blueprint_record)
principal = state.get(binding['principal_record'])
# 4. Save the selected mapping locally; this step creates no Entra objects.
state['blueprint_group'] = blueprint_group
if blueprint:
    state['blueprint'] = blueprint
if principal:
    state['principal'] = principal
save_state()
print({'platform': agent_platform, 'group': blueprint_group, 'membership_approved': AGENT_GROUP_APPROVED, 'blueprint_available': bool(blueprint), 'principal_available': bool(principal)})


### 2.3 Supply the Blueprint appId and create or reuse the Agent Identity

Reuse a known Agent Identity only if its source mapping is known and its actual
Blueprint parent matches. Otherwise enable `CREATE_IDENTITY` only for an
approved new identity. The source agent ID is kept in private state; this Entra
POST does not accept it as a provider-agent binding. With creation selected,
there is no existing-ID prompt; a saved identity is still reused first.

Read back the directory objects and compare actual IDs. This proves the Entra
parent relationship only, not a runtime binding.

In [ ]:
# 1. Use the top-level creation choice, preferring any identity already saved for this agent.
agent_identity = state.get('agent_identity')
if blueprint and principal:
    # 2. Read a known identity; ask for its ID only when reuse mode has no saved value.
    identity_id = agent_identity['id'] if agent_identity else ''
    if not identity_id and not CREATE_IDENTITY:
        identity_id = getpass('Known Test V2 Agent Identity ID, or blank to leave unresolved: ').strip()
    if identity_id:
        agent_identity = request_graph('GET', '/v1.0/servicePrincipals/' + quote(identity_id, safe='') + '/microsoft.graph.agentIdentity')
        assert agent_identity['id'] == identity_id, 'Unexpected identity ID.'
    elif CREATE_IDENTITY:
        # 3. Create only an approved new identity under the Blueprint appId, not its application object ID.
        assert AGENT_GROUP_APPROVED, 'Approve this agent/group membership first.'
        assert not original.get('agentIdentityId'), 'Original already has an identity; reassess before creating one.'
        agent_identity = request_graph('POST', '/v1.0/servicePrincipals/microsoft.graph.agentIdentity', {
            'displayName': 'Test V2 - disposable Agent Identity',
            'agentIdentityBlueprintId': blueprint['appId'],
            'sponsors@odata.bind': [GRAPH + '/v1.0/users/' + str(UUID(state['sponsor_id']))],
            'owners@odata.bind': [GRAPH + '/v1.0/users/' + quote(state['operator_id'], safe='')],
        }, record='agent_identity')
    if agent_identity:
        # 4. Confirm the returned identity's parent and type before saving its actual ID.
        assert agent_identity.get('agentIdentityBlueprintId') == blueprint['appId'], 'Identity has a different Blueprint parent.'
        assert agent_identity.get('servicePrincipalType') == 'ServiceIdentity', 'Unexpected identity type.'
        state['agent_identity'] = agent_identity
        save_state()
print({'agent_identity_available': bool(agent_identity)})

# 5. Re-read the directory objects to confirm their relationship, not runtime authentication.
identity_verified = False
if blueprint and principal and agent_identity:
    checked_blueprint = request_graph('GET', '/v1.0/applications/' + quote(blueprint['id'], safe='') + '/microsoft.graph.agentIdentityBlueprint')
    checked_identity = request_graph('GET', '/v1.0/servicePrincipals/' + quote(agent_identity['id'], safe='') + '/microsoft.graph.agentIdentity')
    assert checked_blueprint['id'] == blueprint['id'] and checked_blueprint['appId'] == blueprint['appId'], 'Blueprint changed.'
    assert checked_identity['id'] == agent_identity['id'] and checked_identity.get('servicePrincipalType') == 'ServiceIdentity', 'Identity changed.'
    assert checked_identity['agentIdentityBlueprintId'] == blueprint['appId'], 'Parent does not match.'
    identity_verified = True
print({'entra_relationship_verified': identity_verified})


## Part 3 - Create or update the agent registration

### 3.1 Get an existing registration when its ID is known

Supply the known companion Registration ID from prior evidence/state, **not**
the original Package ID. This lab already has a retained companion, so reuse it.
There is no documented registration-list/source-ID lookup in the reviewed API.
Missing IDs, 403, or 404 do not establish that no registration exists: stop and
reconcile them rather than automatically POSTing a duplicate.

Choose `CREATE_COMPANION` in the Prep configuration. For an approved first-time creation,
True skips the existing-ID prompt, but 3.2 still requires `FIRST`. A saved
registration is always read and reused, even when creation is enabled.

In [ ]:
# 1. Use the top-level registration choice, then sign in for the required permissions.
association_scopes = DISCOVERY_SCOPES + ENTRA_READ_SCOPES + REGISTRATION_READ_SCOPES
if RUN_WRITES:
    association_scopes += ['AgentRegistration.ReadWrite.All']
sign_in(association_scopes)
# 2. Prefer the saved Registration ID; prompt only when reuse mode needs one.
registration = state.get('registration')
registration_id = registration['id'] if registration else ''
if not registration_id and not CREATE_COMPANION:
    registration_id = getpass('Known Test V2 companion Registration ID, or blank if unresolved: ').strip()
if registration_id:
    # 3. Read by Registration ID, never the original Package ID; failed reads do not trigger creation.
    assert registration_id != state['original_id'], 'Do not use the original Package ID as a Registration ID.'
    registration = request_graph('GET', REGISTRATIONS + '/' + quote(registration_id, safe=''))
    # 4. Check source, platform, and ownership before accepting and saving this companion.
    assert registration['id'] == registration_id and registration.get('sourceAgentId') == state['source_agent_id'], 'Companion/source mismatch.'
    assert registration.get('originatingStore') == agent_platform, 'Wrong source platform.'
    assert state['operator_id'] in registration.get('ownerIds', []) and not registration.get('managedByAppId'), 'Expected the approved user-owned companion.'
    state['registration'] = registration
    save_state()
print({'known_companion_available': bool(registration)})


### 3.2 Update the known registration, or explicitly create the first companion

Supply **Blueprint `appId` + Agent Identity `id` + exact source agent ID**.
For an existing registration, verify source/owner first and PATCH only the
identity fields; never overwrite conflicting links. For an approved first-time
companion, set `CREATE_COMPANION = True` and confirm `FIRST`. An unavailable
registration is not automatically a first-time registration.

Keep creation off for the current retained companion. The original Registry
Sync package is never mutated. A new registration also needs its display name,
platform, creator/owner, and original source timestamps; the three IDs alone
are not a complete POST body. Do not copy `managedByAppId`.

POST returns 201; PATCH returns 200 with an empty body. String acceptance does
not prove semantic identity resolution or enrichment of the synchronized record.

In [ ]:
# 1. Require verified Entra objects and approved agent membership before associating identities.
if identity_verified:
    if RUN_WRITES:
        assert AGENT_GROUP_APPROVED, 'Approve this agent/group membership first.'
    # 2. Re-read the original and stop if its identity status changed; build links from actual Entra IDs.
    current_original = request_graph('GET', PACKAGES + '/' + quote(state['original_id'], safe=''))
    assert not current_original.get('agentIdentityId'), 'Original now has an identity; reassess native association.'
    identity_links = {'agentIdentityBlueprintId': blueprint['appId'], 'agentIdentityId': agent_identity['id']}
    if registration:
        # 3. Recheck the companion and PATCH missing links only; do not overwrite conflicting associations.
        current = request_graph('GET', REGISTRATIONS + '/' + quote(registration['id'], safe=''))
        assert current.get('sourceAgentId') == state['source_agent_id'] and current.get('originatingStore') == agent_platform, 'Source changed.'
        assert state['operator_id'] in current.get('ownerIds', []) and not current.get('managedByAppId'), 'Ownership changed.'
        assert all(current.get(k) in (None, '', v) for k, v in identity_links.items()), 'Different identity links already exist; do not overwrite.'
        if all(current.get(k) == v for k, v in identity_links.items()):
            print({'association': 'already stored; no write needed'})
        else:
            association_result = request_graph('PATCH', REGISTRATIONS + '/' + quote(registration['id'], safe=''), identity_links, record='association_update')
    elif CREATE_COMPANION:
        # 4. Confirm first creation and preserve source timestamps; never POST merely because an ID was lost.
        assert getpass('Type FIRST to confirm a companion has never been created for this source: ') == 'FIRST', 'Creation not confirmed.'
        for timestamp in (source.get('CreatedDateTime'), source.get('LastModifiedDateTime')):
            assert isinstance(timestamp, str) and datetime.fromisoformat(timestamp).tzinfo is not None, 'Keep valid original ISO timestamps.'
        registration = request_graph('POST', REGISTRATIONS, {
            'displayName': 'Test V2 - identity companion',
            'description': 'Disposable companion; original Registry Sync record retained',
            'createdBy': state['operator_id'], 'ownerIds': [state['operator_id']],
            'sourceAgentId': state['source_agent_id'], 'originatingStore': agent_platform,
            'sourceCreatedDateTime': source['CreatedDateTime'],
            'sourceLastModifiedDateTime': source['LastModifiedDateTime'],
            **identity_links,
        }, record='registration')
    else:
        print({'association': 'blocked - known companion required'})
else:
    print({'association': 'blocked - actual verified Entra objects required'})


### 3.3 Read back and compare registration and inventory

Compare stored links to the actual Entra objects, then inspect original and
companion packages. Registration ID is only a candidate Package ID for this
observed API-created companion, not a universal ID rule. A 404 remains unavailable.
Saved successful writes are not repeated. Subsequent sync behavior and runtime
token use remain separate, unperformed experiments.

In [ ]:
# 1. Compare the registration's stored links with the actual Entra IDs; matching strings are not resolution proof.
links_match = False
if registration and blueprint and agent_identity:
    checked_registration = request_graph('GET', REGISTRATIONS + '/' + quote(registration['id'], safe=''))
    assert checked_registration['id'] == registration['id'] and checked_registration.get('sourceAgentId') == state['source_agent_id'], 'Unexpected registration.'
    links_match = checked_registration.get('agentIdentityBlueprintId') == blueprint['appId'] and checked_registration.get('agentIdentityId') == agent_identity['id']
    state['registration'] = checked_registration
    save_state()
print({'registration_identity_fields_match': links_match, 'semantic_resolution_proven': False})

# 2. Refresh inventory and compare the original's identity field with the saved pre-write baseline.
packages_after = list_packages()
original_after = request_graph('GET', PACKAGES + '/' + quote(state['original_id'], safe=''))
assert read_source(original_after)['SourceAgentId'] == state['source_agent_id'], 'Original source changed.'
original_identity_unchanged = original_after.get('agentIdentityId') == state['original_before'].get('agentIdentityId')
# 3. Try Registration ID as a Package ID only for this known companion, then confirm its source.
companion_package = None
if registration:
    companion_package = request_graph('GET', PACKAGES + '/' + quote(registration['id'], safe=''), expected=(200, 404))
    if companion_package:
        assert companion_package['id'] == registration['id'] and companion_package['id'] != state['original_id'], 'Unexpected package correlation.'
        assert read_source(companion_package)['SourceAgentId'] == state['source_agent_id'], 'Companion source mismatch.'
# 4. Save comparison details privately and report only observed field-level results.
state['original_after'] = original_after
state['companion_package'] = companion_package
save_state()
print({'original_identity_unchanged': original_identity_unchanged, 'companion_package_available': bool(companion_package), 'companion_identity_field_matches': bool(companion_package and agent_identity and companion_package.get('agentIdentityId') == agent_identity['id']), 'semantic_association': 'inconclusive'})

# 5. Stop on unresolved writes; otherwise read whether another association write would be necessary.
assert not state.get('pending_write'), 'A write outcome remains unknown; stop and reconcile it.'
no_write_needed = False
if registration and identity_verified:
    latest = request_graph('GET', REGISTRATIONS + '/' + quote(registration['id'], safe=''))
    no_write_needed = latest.get('agentIdentityBlueprintId') == blueprint['appId'] and latest.get('agentIdentityId') == agent_identity['id'] and latest.get('sourceAgentId') == state['source_agent_id']
print({'association_write_needed': not no_write_needed, 'original_identity_unchanged': original_identity_unchanged, 'subsequent_sync_observed': False})

# 6. Directory and registration metadata do not demonstrate provider-runtime integration.
print({'runtime_binding': 'not tested', 'provider_runtime_changed': False})


## Cleanup and limits

Default: retain everything (`DELETE_OBJECT = None`). No provider agent or
Registry Sync connection is deleted. The earlier retention decision is unchanged.

For a separately approved cleanup, select one `registration`, `agent_identity`,
or `blueprint`, enable writes, and confirm the dependency review. Only an object
recorded in `created_here` can be deleted. A retained companion blocks deletion
of its identity; an existing child blocks deletion of its Blueprint.

`CLEANUP_PLATFORM` / `CLEANUP_GROUP` select a Part 1 Blueprint independently of
the agent's group, so every newly created group can be reviewed and retired.
Blueprint deletion can cascade to its principal/children. Review dependents in
Entra first, delete/confirm children before their parent, and confirm the
principal's retirement in the portal. If only a principal was created for a
reused Blueprint, have the owner retire that specific principal in Entra after
dependency review; do not delete the reused Blueprint to remove it.

Keep `state.json` while IDs or pending outcomes are needed. On completion,
clear outputs and restart/stop the kernel. Remove only individual no-longer-needed
files: `candidates.json`, `state.tmp`, an older `inventory-details.json` if retained, eventually
`state.json`, and the exact working notebook/checkpoint or preserved pre-restructure
notebook. Do not remove earlier HTTP evidence, shared caches, or the whole workspace.
Retain or remove this lab's `.venv`/`uv.lock` only when finished.

In [ ]:
cleanup_platform = CLEANUP_PLATFORM or agent_platform
cleanup_group = CLEANUP_GROUP or blueprint_group
if DELETE_OBJECT is None:
    print({'cleanup': 'retained - no delete requested'})
else:
    assert RUN_WRITES and CONFIRMED_NO_DEPENDENTS, 'Deletion and dependency review must be explicitly approved.'
    assert DELETE_OBJECT in ('registration', 'agent_identity', 'blueprint'), 'Unsupported cleanup target.'
    record_key = bindings[cleanup_platform][cleanup_group]['blueprint_record'] if DELETE_OBJECT == 'blueprint' else DELETE_OBJECT
    target = state[record_key]
    assert state.get('created_here', {}).get(record_key) == target['id'], 'Do not delete reused or pre-existing objects.'
    deleted = state.setdefault('confirmed_deleted', [])
    assert record_key not in deleted, 'Deletion already confirmed.'
    if DELETE_OBJECT == 'agent_identity':
        assert not state.get('registration') or 'registration' in deleted, 'The companion is still retained.'
    if DELETE_OBJECT == 'blueprint':
        identity_depends = state.get('agent_identity', {}).get('agentIdentityBlueprintId') == target['appId']
        assert not identity_depends or 'agent_identity' in deleted, 'The child identity is still retained.'
        registration_depends = state.get('registration', {}).get('agentIdentityBlueprintId') == target['appId']
        assert not registration_depends or 'registration' in deleted, 'The companion is still retained.'
    routes = {
        'registration': (REGISTRATIONS + '/' + quote(target['id'], safe=''), 'AgentRegistration.ReadWrite.All'),
        'agent_identity': ('/v1.0/servicePrincipals/' + quote(target['id'], safe='') + '/microsoft.graph.agentIdentity', 'AgentIdentity.DeleteRestore.All'),
        'blueprint': ('/v1.0/applications/' + quote(target['id'], safe='') + '/microsoft.graph.agentIdentityBlueprint', 'AgentIdentityBlueprint.DeleteRestore.All'),
    }
    delete_path, permission = routes[DELETE_OBJECT]
    sign_in(association_scopes + [permission])
    current_original = request_graph('GET', PACKAGES + '/' + quote(state['original_id'], safe=''))
    assert not current_original.get('agentIdentityId'), 'Original may now depend on the identity; do not delete it.'
    deletion_record = 'deleted_' + record_key
    if deletion_record not in state.get('completed_writes', []):
        deletion_result = request_graph('DELETE', delete_path, record=deletion_record)
    remaining = request_graph('GET', delete_path, expected=(200, 404))
    assert remaining is None, 'Object is still addressable; stop further cleanup.'
    deleted.append(record_key)
    save_state()
    print({'cleanup': 'selected object is no longer addressable', 'cascade': 'check separately in the portal'})
access_token = None
access_token_scopes = set()
access_token_expires_at = 0
auth_account = None
auth_client = None
print({'token_reference_cleared': True, 'next': 'restart or stop the kernel'})
